# From text to vectors to next word — build it from scratch

**Large Language Models for Generative AI · practice topic 01**

Open `explainer.html` first. This notebook walks the same path, but you write the key lines.

## The problem

A model cannot read words — only numbers. Take a tiny six-sentence story and turn it, step by step, into the things
an LLM actually works with:

| # | Step | The idea |
|---|------|----------|
| 1 | **Tokenize** | Cut text into pieces |
| 2 | **Vocabulary + IDs** | Give every piece a number; handle words you have never seen (OOV) |
| 3 | **One-hot** | The crudest number-vector for a word |
| 4 | **Embedding** | A short learned vector; looking it up is a row-pick |
| 5 | **Cosine similarity** | Are two word-vectors pointing the same way? |
| 6 | **Next-word model** | Count which word follows which, then generate text one word at a time |

Only `numpy` and `re` are needed. Lines marked `# TODO` are yours; everything else is done.

---
## Cell 0 — Setup and the corpus

Nothing to write. A **corpus** is just the pile of text you learn from. Ours has 38 tokens.

In [ ]:
import re
import numpy as np

np.set_printoptions(precision=3, suppress=True)

CORPUS = ("the cat sat on the mat . the dog sat on the rug . the cat chased the dog . "
          "the dog chased the cat . a cat saw a dog . the dog saw the cat .")
print(CORPUS)

---
## Cell 1 — Tokenize  ·  *explainer section 2*

A **token** is the unit a model reads. Real LLMs use sub-word pieces; we use the simplest version: lowercase, then
take every run of letters as a token, and keep the full stop as its own token.

Tool: `re.findall(pattern, text)` returns every match as a list. The pattern `r"[a-z]+|[.]"` means
*"a run of letters, OR a single dot"*.

In [ ]:
def tokenize(text):
    """Return the list of tokens in `text`."""
    # TODO 1: lowercase the text, then return re.findall(r"[a-z]+|[.]", ...) of it.
    return None


tokens = tokenize(CORPUS)
assert tokens is not None, "tokenize returned None -- fill in TODO 1"
print(tokens[:10])

assert len(tokens) == 38, "expected 38 tokens, got {}".format(len(tokens))
assert tokens[0] == "the" and tokens[6] == "."
print("check passed --", len(tokens), "tokens")

---
## Cell 2 — Vocabulary, IDs and OOV  ·  *explainer section 3*

The **vocabulary** is the fixed list of distinct tokens. Each gets an integer **ID** — its position in the sorted list.

**OOV** (out-of-vocabulary) = a word not in the list. We reserve one extra entry, `<unk>`, and send every stranger there.
(Modern LLMs use byte-level pieces so this never happens — which is exactly why it is worth seeing the old problem once.)

In [ ]:
vocab = sorted(set(tokens))
# TODO 2a: build word_to_id, a dict mapping each word in `vocab` to its position (0, 1, 2, ...).
#          Hint: a dict comprehension with enumerate(vocab).
word_to_id = None

# Done for you: add the catch-all entry at the end.
word_to_id["<unk>"] = len(word_to_id)
V = len(word_to_id)


def encode(words):
    """Turn a list of words into a list of IDs. Unknown words become <unk>."""
    # TODO 2b: for each word, look it up in word_to_id; if it is missing use word_to_id["<unk>"].
    #          Hint: dict.get(key, default)
    return None


print(word_to_id)
print(encode(["the", "cat", "sat"]))
print(encode(["the", "giraffe", "sat"]))

assert V == 12, "11 real words + <unk> = 12, got {}".format(V)
assert encode(["the", "cat"]) == [word_to_id["the"], word_to_id["cat"]]
assert encode(["giraffe"]) == [word_to_id["<unk>"]], "unseen words must map to <unk>"
print("check passed -- |V| =", V)

---
## Cell 3 — One-hot vectors  ·  *explainer section 4*

A **one-hot vector** has length `|V|`: all zeros except a single `1` at the word's ID.

It names the word and says nothing about meaning — check the last line: every pair of different words is exactly as
far apart as every other pair.

In [ ]:
def one_hot(word_id, size):
    """Return a numpy vector of length `size`, all zeros except a 1 at `word_id`."""
    # TODO 3: make a zero vector of length `size` (np.zeros) and set the one slot to 1.
    v = None
    return v


cat, dog, mat = (one_hot(word_to_id[w], V) for w in ("cat", "dog", "mat"))
print("cat:", cat)

assert cat.sum() == 1 and cat[word_to_id["cat"]] == 1
assert cat @ dog == 0 and cat @ mat == 0, "different words must share no 1s"
print("check passed -- cat.dog =", cat @ dog, " cat.mat =", cat @ mat, "(no similarity information at all)")

---
## Cell 4 — Embedding = a row lookup  ·  *explainer section 4*

An **embedding matrix** `E` has one row per word and `d` columns (`d` = the embedding size). A word's embedding is its
row. The surprise: **multiplying a one-hot vector by `E` just picks that row** — so real models skip the multiplication
and index directly.

`E` here is random. In a real model, training nudges those numbers until similar words end up with similar rows.

In [ ]:
d = 4
rng = np.random.default_rng(0)
E = rng.normal(size=(V, d))          # |V| x d, random for now

i = word_to_id["dog"]

# TODO 4a: pick the embedding of "dog" by multiplying one_hot(i, V) by E  (use @).
via_matmul = None
# TODO 4b: get the same thing by simply indexing row i of E.
via_lookup = None

# TODO 4c: how many learned numbers does E hold? (one expression using V and d)
n_params = None

print("dog embedding:", via_lookup)
assert np.allclose(via_matmul, via_lookup), "the two routes must give the same vector"
assert n_params == V * d
print("check passed --", n_params, "parameters. Llama 3 8B: 128,256 x 4,096 = 525M.")

---
## Cell 5 — Cosine similarity  ·  *explainer section 6*

How close are two vectors in *direction*?

`cos(a, b) = (a · b) / (|a| · |b|)` — dot product divided by both lengths. 1 = same direction, 0 = unrelated.

Dividing by the lengths means a long vector and a short one pointing the same way still score 1.

In [ ]:
def cosine(a, b):
    """Cosine similarity of two 1-D numpy vectors."""
    # TODO 5: dot product divided by (norm of a times norm of b).  np.linalg.norm gives a vector's length.
    return None


assert abs(cosine(np.array([1, 0]), np.array([1, 0])) - 1) < 1e-9, "same direction -> 1"
assert abs(cosine(np.array([1, 0]), np.array([0, 1]))) < 1e-9, "at right angles -> 0"
assert abs(cosine(np.array([1, 2]), np.array([10, 20])) - 1) < 1e-9, "length must not matter"
print("check passed")
print("one-hot cat vs dog:", cosine(cat, dog), " <- all one-hot pairs are 0")

---
## Cell 6 — Meaning from company: a static embedding  ·  *explainer section 5*

One-hot says nothing. So where does meaning come from? From **the company a word keeps**: words used in the same
spots (`cat` and `dog` both follow `the`, both precede `sat`/`chased`/`saw`) should get similar vectors.

Build a **co-occurrence matrix**: row = a word, column = a neighbour, cell = how many times that neighbour appeared
directly to the left or right of the word. Each row is then a (crude) static embedding of its word.

In [ ]:
co = np.zeros((V, V))

for pos, word in enumerate(tokens):
    for nb in (pos - 1, pos + 1):                  # left and right neighbour positions
        if 0 <= nb < len(tokens):
            # TODO 6: add 1 to the cell at [row of `word`, column of the neighbour token].
            #         Hint: use word_to_id[...] for both indexes.
            pass


def sim(w1, w2):
    return cosine(co[word_to_id[w1]], co[word_to_id[w2]])


for a, b in [("cat", "dog"), ("mat", "rug"), ("cat", "mat"), ("cat", "the")]:
    print("{:>4} vs {:<4} {:.3f}".format(a, b, sim(a, b)))

assert co.sum() == 2 * (len(tokens) - 1), "every neighbouring pair is counted twice"
assert sim("cat", "dog") > sim("cat", "the"), "animals should sit closer than animal vs 'the'"
print("check passed")

**Read the numbers.** `cat`/`dog` and `mat`/`rug` score 1.0 — they appear in exactly the same slots. Nobody told the
program they are animals or furniture; it only watched where they stand. That is the whole idea behind word2vec-style
embeddings, and it is *static*: `bank` would get one vector whatever the sentence.

In [ ]:
# Your turn: which word is most similar to "sat"? Fill in, then run.
# TODO 7: loop over the real words (not "<unk>", not "sat" itself), call sim("sat", w), keep the best.
best_word, best_score = None, -1

print(best_word, best_score)

---
## Cell 7 — A language model: which word comes next?  ·  *explainer section 7*

A **language model** gives a probability for the next token given the previous one(s). The simplest one counts:

1. `counts[a, b]` = how many times word `b` came straight after word `a`.
2. Divide each row by its total → each row becomes probabilities that sum to 1.

In [ ]:
counts = np.zeros((V, V))
for prev, nxt in zip(tokens, tokens[1:]):
    # TODO 8a: add 1 to counts[row of prev, column of nxt].
    pass

# TODO 8b: turn counts into probabilities P: divide every row by its own sum.
#          Hint: counts.sum(axis=1, keepdims=True).  Rows with sum 0 give nan -- handled on the next line.
with np.errstate(invalid="ignore"):
    P = None
P = np.nan_to_num(P)

row = P[word_to_id["the"]]
print({w: round(float(row[i]), 2) for w, i in word_to_id.items() if row[i] > 0})

assert abs(row.sum() - 1) < 1e-9, "a probability row must sum to 1"
assert abs(row[word_to_id["cat"]] - 0.4) < 1e-9 and abs(row[word_to_id["mat"]] - 0.1) < 1e-9
print("check passed -- after 'the': cat 40%, dog 40%, mat 10%, rug 10%")

---
## Cell 8 — Autoregressive generation  ·  *explainer section 7*

**Autoregressive** = produce one token, add it to the text, use it to produce the next. Chat models do exactly
this loop, just with a far bigger model than our table.

Greedy decoding: always pick the most probable next word.

In [ ]:
id_to_word = {i: w for w, i in word_to_id.items()}


def generate(start, max_tokens=12):
    """Greedy autoregressive generation starting from the word `start`."""
    out = [start]
    while len(out) < max_tokens and out[-1] != ".":
        # TODO 9: take the probability row for the last word in `out`, pick the ID with the
        #         highest probability (np.argmax), turn it back into a word with id_to_word, append it.
        break
    return out


print(" ".join(generate("the")))
print(" ".join(generate("a")))

result = generate("a")
assert result[0] == "a" and len(result) > 1, "generate should add at least one word"
print("check passed")

---
## Cell 9 — Sampling instead of always-the-best  ·  *optional*

Greedy gives the same sentence every time. Real chatbots **sample**: pick the next word at random, weighted by its
probability, so "cat" and "dog" (40% each after `the`) both get their turn.

In [ ]:
rng = np.random.default_rng(1)


def generate_sampled(start, max_tokens=12):
    out = [start]
    while len(out) < max_tokens and out[-1] != ".":
        probs = P[word_to_id[out[-1]]]
        # TODO 10 (optional): draw one ID with rng.choice(V, p=probs) and append its word.
        break
    return out


for _ in range(3):
    print(" ".join(generate_sampled("the")))

---
## Cell 10 — Say it in words

Nothing to run. Answer in your own words (the same questions are in `guide.md`):

1. Why is a one-hot vector useless for finding that `cat` and `dog` are related?
2. What is the difference between a *token* and a *word*, and why does `|V|` matter for cost?
3. Why does `generate("the")` always give the same sentence, and what does sampling change?
4. Our model only looks at **one** previous word. What kinds of sentence will it get wrong?